# Loan Applications: Cleaning and Exploratory Analysis

**Author:** Maria Pupysheva · **Date:** October 2026

**Data:** Loan Prediction dataset (Kaggle): 614 home loan applications with applicant details,
loan details and the lender's decision (`Loan_Status`). Source: [Kaggle](https://www.kaggle.com/datasets/burak3ergun/loan-data-set)

**Goal:** clean the data, engineer affordability features (such as repayment-to-income) and
explore which factors are linked to loan approval. This is practice for the data-preparation
part of the Westpac Hackathon project.

**Note:** `Loan_Status` records whether the loan was **approved**, not whether it was repaid.
Findings describe the lender's past decisions, not repayment risk.

**Tools:** Python 3.12 · pandas · NumPy · Matplotlib

**Contents**
1. Setup · 2. Data overview · 3. Cleaning · 4. Feature engineering · 5. Exploratory analysis · 6. Summary

## 1. Setup
Load the raw CSV. The data file is kept in `data/` and is not uploaded to GitHub.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)   # show all columns in wide tables

In [2]:
loans_raw = pd.read_csv("../data/loan_data_set.csv")
loans_raw.shape

(614, 13)

## 2. Data overview
A first look at the structure, types, missing values and distributions before any cleaning.

### Q1. Structure
How many applications and columns are there, and what does one row represent?

In [3]:
print(f"Applications: {len(loans_raw)}, columns: {loans_raw.shape[1]}")
loans_raw.head()

Applications: 614, columns: 13


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


**Observation:** 614 loan applications with 13 columns: one row per application, with applicant
details (gender, marital status, dependents, education, employment, income), loan details
(amount, term), credit history, property area and the lender's decision (`Loan_Status`).

### Q2. Column types
Which columns are numbers and which are text, and is any column stored as the wrong type?

In [4]:
loans_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    str    
 1   Gender             601 non-null    str    
 2   Married            611 non-null    str    
 3   Dependents         599 non-null    str    
 4   Education          614 non-null    str    
 5   Self_Employed      582 non-null    str    
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    str    
 12  Loan_Status        614 non-null    str    
dtypes: float64(4), int64(1), str(8)
memory usage: 62.5 KB


In [5]:
display(loans_raw["Dependents"].unique())
display(loans_raw["Credit_History"].unique())

<StringArray>
['0', '1', '2', '3+', nan]
Length: 5, dtype: str

array([ 1.,  0., nan])

**Observation:**
- `Dependents` is text because of the value `3+`; it needs converting to a number
  (treating `3+` as 3, meaning "3 or more").
- `Credit_History` is stored as float only because it has missing values. Its values are
  1 (meets the lender's credit guidelines) and 0 (does not), so it is really a yes/no flag.

### Q3. Missing values
Which columns have missing values, how many, and what share of rows?

In [6]:
missing = (
    loans_raw.isna().sum()
    .to_frame("missing")
    .assign(missing_pct=lambda df: df["missing"] / len(loans_raw))
    .query("missing > 0")
    .sort_values("missing", ascending=False)
)
missing.style.format({"missing_pct": "{:.1%}"})

,missing,missing_pct
Credit_History,50,8.1%
Self_Employed,32,5.2%
LoanAmount,22,3.6%
Dependents,15,2.4%
Loan_Amount_Term,14,2.3%
Gender,13,2.1%
Married,3,0.5%


**Observation:** 7 columns have missing values, from 0.5% (`Married`) to 8.1% (`Credit_History`).
`Credit_History` matters most: it is the most lending-relevant column and has the most missing
values (50), so how it is handled can change the conclusions. `Self_Employed` is next (32, 5.2%).

### Q4. Numeric columns
Distribution of incomes, loan amount and loan term.

In [7]:
numeric_cols = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Loan_Amount_Term"]
loans_raw[numeric_cols].describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
ApplicantIncome,614.0,5403.5,6109.0,150.0,2877.5,3812.5,5795.0,81000.0
CoapplicantIncome,614.0,1621.2,2926.2,0.0,0.0,1188.5,2297.2,41667.0
LoanAmount,592.0,146.4,85.6,9.0,100.0,128.0,168.0,700.0
Loan_Amount_Term,600.0,342.0,65.1,12.0,360.0,360.0,360.0,480.0


In [8]:
loans_raw["Loan_Amount_Term"].value_counts(dropna=False).sort_index()

Loan_Amount_Term
12.0       1
36.0       2
60.0       2
84.0       4
120.0      3
180.0     44
240.0      4
300.0     13
360.0    512
480.0     15
NaN       14
Name: count, dtype: int64

**Observation:**
- Incomes are strongly right-skewed: applicant income has a median of 3,812 but a maximum of
  81,000; the mean (5,403) is pulled up by a few very high earners, so medians are more
  representative.
- At least 25% of applicants have no co-applicant (co-applicant income of 0).
- `LoanAmount` is most likely in thousands (median 128, minimum 9); incomes are most likely
  monthly. The dataset does not document units, so these are assumptions.
- `Loan_Amount_Term` is in months: 360 months (30 years) is by far the most common (512 of 600),
  with a few unusual short terms down to 12 months.

### Q5. Categorical columns
Values and counts of each text column, including missing values.

In [9]:
categorical_cols = ["Gender", "Married", "Dependents", "Education", "Self_Employed", "Property_Area"]

for col in categorical_cols:
    display(loans_raw[col].value_counts(dropna=False).rename_axis(col).reset_index(name="applications"))

,Gender,applications
0,Male,489
1,Female,112
2,NaN,13


,Married,applications
0,Yes,398
1,No,213
2,NaN,3


,Dependents,applications
0,0,345
1,1,102
2,2,101
3,3+,51
4,NaN,15


,Education,applications
0,Graduate,480
1,Not Graduate,134


,Self_Employed,applications
0,No,500
1,Yes,82
2,NaN,32


,Property_Area,applications
0,Semiurban,233
1,Urban,202
2,Rural,179


In [10]:
print(f"Loan_ID is unique: {loans_raw['Loan_ID'].is_unique}")

Loan_ID is unique: True


**Observation:** Categories are spelled consistently and `Loan_ID` is unique (no duplicates).
Groups are unbalanced: 80% of known applicants are male, 65% are married and only 14% are
self-employed, so approval rates for small groups will be less reliable. `Dependents` should be
treated as a number (0, 1, 2, 3+), not as free text.

### Q6. The target
What share of applications was approved?

In [11]:
status = (
    loans_raw["Loan_Status"]
    .value_counts()
    .rename_axis("loan_status")
    .reset_index(name="applications")
)
status["share"] = status["applications"] / status["applications"].sum()
status.style.format({"share": "{:.1%}"}).hide(axis="index")

loan_status,applications,share
Y,422,68.7%
N,192,31.3%


**Observation:** 68.7% of applications were approved and 31.3% rejected. The classes are
unbalanced: a rule that approves everyone would already be right 68.7% of the time, so any
model must beat that baseline, and groups should be compared by approval rate, not by counts.

### Data quality notes
These problems form the cleaning plan for section 3.

1. Missing values in 7 columns, most in `Credit_History` (8.1%) and `Self_Employed` (5.2%).
2. `Dependents` is text (`3+`); convert to a number.
3. `Credit_History` is a 0/1 flag stored as float; convert after handling missing values.
4. Incomes are heavily right-skewed with extreme values (up to 81,000): check for outliers.
5. Units are undocumented: `LoanAmount` assumed in thousands, incomes assumed monthly.
6. A few unusual loan terms (12–84 months) to review.